# Fingerprints to SMILES amb MolForge
Transforma una taula amb **fingerprints** en **SMILES** fent servir MolForge.

**Entrada**: CSV amb columna `fingerprints_input_ECPF4`.

**Sortida**: CSV amb columnes `fingerprints_input_ECFP4` i `SMILES_output_ECPF4`.

## Imports

In [1]:
# Per definir l'arrel del projecte
import os

# Pandas pels dataframes
import pandas as pd
import numpy as np # pels NaN

# Per cridar MolForge i guardar-ne l'output
from MolForge import main as molforge_main
from MolForge import predict as mf
import sys
import io
from contextlib import redirect_stdout

## Inputs (part a editar)

Arrel del projecte

In [2]:
#os.chdir("/export/home/ddiestre/MolForge_Testing")
#os.chdir("/mnt/c/Users/david/Desktop/Uni/MolForge_Testing")
os.chdir("/mnt/d/MolForge_Testing")

Paràmetres de MolForge

In [3]:
FP_NAME = "ECFP4"
MODEL_TYPE = "smiles"  # ["smiles", "selfies"]
DECODE = "greedy"  # ["greedy", "beam"]
CHECKPOINT_NAME = "ECFP4_smiles_checkpoint.pth"

Fitxer de fingerprints preprocessat (path a partir de MolForge_Testing/)

In [4]:
#input_path = "data/MolForge_input/MolForge_MFinput_2000_ECFP4_noise/MolForge_MFinput_2000_ECFP4_noise0_0_noise1_20.csv"
input_path = "data/MolForge_input/CoCoGraph_MFinput_2000_novel_noise/CoCoGraph_MFinput_2000_novel_noise0_0_6_noise1_0_6.csv"
#input_path = "data/MolForge_input/CoCoGraph_MFinput_2000_lt70atoms.csv"
#input_path = "data/MolForge_input/PubChem_MFinput_2000_CID-SMILES-filtered-lt70.csv"

in_col_name = "fingerprints_input_" + FP_NAME

Fitxer en que guardar l'output (path a partir de MolForge_Testing/)

In [5]:
#output_root = "data/MolForge_output/MolForge_MFoutput_2000_ECFP4_noise/"
#output_file = "MolForge_MFoutput_2000_ECFP4_noise0_0_noise1_20.csv"

output_root = "data/MolForge_output/CoCoGraph_MFoutput_2000_novel_noise/"
output_file = "CoCoGraph_MFoutput_2000_novel_noise0_0_6_noise1_0_6.csv"

#output_root = "data/MolForge_output/CoCoGraph_MFoutput_2000_lt70atoms_noise/"
#output_file = "CoCoGraph_MFoutput_2000_lt70atoms.csv"

#output_root = "data/MolForge_output/"
#output_file = "PubChem_MFoutput_2000_CID-SMILES-filtered-lt70.csv"

out_col_name = "SMILES_output_" + FP_NAME
output_path = output_root + output_file
os.makedirs(output_root, exist_ok=True)  # crea tota la ruta si no existeix

## 1. Lectura del fitxer

In [6]:
# Lectura del fitxer
df = pd.read_csv(input_path, sep = ',', index_col = 0)
df.head(5)

,SMILES_input,fingerprints_input_ECFP4
id,,
1,OCCCCN1CCCC1,80 222 422 473 541 646 653 740 743 774 807 810...
2,N#CN(CC(N)=O)c1ccc(Cl)cc1-c1cccnn1,46 73 80 140 165 212 216 218 281 322 352 353 3...
3,Cc1cc[n+](C)cc1-c1ccccc1,106 222 263 350 352 389 391 440 463 482 522 67...
4,COC(=O)NC1CCN(C(=O)CCCCC(=O)N2CCCCC2)C1,2 16 80 129 194 202 222 369 387 433 505 526 65...
5,COc1cccc(S(=O)(=O)NCC(=O)OC=CC2=CC=C2C(=O)Nc2c...,13 80 102 155 191 198 261 263 314 317 319 322 ...


## 2. Execució de Molforge

In [7]:
# Estat global per guardar model i args després de la primera inferència
_MF_STATE = {
    "model": None,
    "args": None,
}


def carregar_model_i_args_un_cop(fp_name, model_type, checkpoint, decode):
    """
    Fa una crida a molforge_main() amb una fingerprint dummy, però
    amb un 'hook' sobre mf.inference per capturar el `model` i `args`
    que MolForge utilitza internament. El model i els args es carreguen
    una sola vegada i es guarden a _MF_STATE.
    """

    # Si ja l'hem carregat abans en aquesta sessió de notebook, el reutilitzem
    if _MF_STATE["model"] is not None and _MF_STATE["args"] is not None:
        return _MF_STATE["model"], _MF_STATE["args"]

    # 1) Guardem la funció d'inferència original
    original_inference = mf.inference

    def hook_inference(*f_args, **f_kwargs):
        """
        Hook que es cridarà en lloc d'mf.inference la PRIMERA vegada.
        Guarda model i args a _MF_STATE i després delega a l'original.
        La signatura exacta no importa perquè fem servir *args, **kwargs.
        """
        # Per com està escrit predict.py, la signatura és:
        # inference(model, input_sentence, method, args, return_attn=False)
        # → model = f_args[0], args = f_args[3]
        model = f_args[0]
        args = f_args[3]

        if _MF_STATE["model"] is None:
            _MF_STATE["model"] = model
            _MF_STATE["args"] = args

        # Fem la inferència normalment (una sola vegada, amb input dummy)
        return original_inference(*f_args, **f_kwargs)

    # 2) Substituïm temporalment mf.inference pel hook
    mf.inference = hook_inference

    # 3) Preparem sys.argv per a una única crida "dummy"
    original_argv = sys.argv
    sys.argv = [
        "",
        f"--fp={fp_name}",
        f"--model_type={model_type}",
        "--input=1",                # input dummy, només per activar main()
        f"--checkpoint={checkpoint}",
        f"--decode={decode}",
    ]

    # 4) Cridem molforge_main amb la sortida redirigida (logs → silenci)
    buf = io.StringIO()
    try:
        with redirect_stdout(buf):
            molforge_main()
    finally:
        # Restaurem sys.argv i la funció d'inferència original
        sys.argv = original_argv
        mf.inference = original_inference

    # 5) Comprovem que hem capturat model i args
    if _MF_STATE["model"] is None or _MF_STATE["args"] is None:
        raise RuntimeError(
            "No s'ha pogut capturar el model i/o els args des de MolForge. "
            "Revisa si MolForge.predict.inference s'està cridant dins de main()."
        )

    return _MF_STATE["model"], _MF_STATE["args"]

In [8]:
def run_molforge_batch_wrapper(
    df,
    fp_col,
    fp_name,
    model_type,
    checkpoint_name,
    decode,
):
    """
    Utilitza MolForge amb el model carregat UNA sola vegada.
    - Carrega model+args amb `carregar_model_i_args_un_cop(...)`
    - Fa un bucle per totes les fingerprints vàlides
    - Crida directament `mf.inference(model, fp_str, args.decode, args, ...)`
      capturant el stdout per extreure "Result: ..."
    - Afegeix una columna `SMILES_out` al DataFrame.
    """

    df_out = df.copy()
    df_out[out_col_name] = np.nan

    # 1) Files vàlides (descartem NaN i "InvalidSMILE")
    mask_valid = (~df_out[fp_col].isna()) & (df_out[fp_col] != "InvalidSMILE")
    idx_valid = df_out.index[mask_valid]
    n_valid = len(idx_valid)

    if n_valid == 0:
        print("No hi ha fingerprints vàlides per passar a MolForge.")
        return df_out

    print(
        f"Passant MolForge per {n_valid} fingerprints vàlides "
        f"(model carregat una sola vegada)..."
    )

    # 2) Carreguem model i args UNA sola vegada (i es cachegen a _MF_STATE)
    model, args = carregar_model_i_args_un_cop(
        fp_name=fp_name,
        model_type=model_type,
        checkpoint=checkpoint_name,
        decode=decode,
    )

    # 3) Bucle sobre les fingerprints vàlides
    n_done = 0
    for idx in idx_valid:
        fp_str = str(df_out.at[idx, fp_col]).strip()

        # Capturem stdout de la inferència per aquesta fingerprint
        buf = io.StringIO()
        with redirect_stdout(buf):
            # IMPORTANT: fem servir la funció d'inferència original del mòdul
            mf.inference(model, fp_str, args.decode, args)

        out = buf.getvalue()

        # Busquem la línia "Result: ..."
        pred_smi = np.nan
        for line in out.splitlines():
            line = line.strip()
            if line.startswith("Result:"):
                pred_smi = line.split("Result:", 1)[1].strip().replace(" ", "")
                break

        df_out.at[idx, out_col_name] = pred_smi

        n_done += 1
        print(f"\r[{n_done}/{n_valid}]", end="", flush=True)

    print()  # salt de línia final
    return df_out

In [9]:
df = run_molforge_batch_wrapper(
    df=df,
    fp_col=in_col_name,      # p. ex. "fingerprints_input_ECFP4"
    fp_name=FP_NAME,         # "ECFP4"
    model_type=MODEL_TYPE,   # "smiles"
    checkpoint_name=CHECKPOINT_NAME,
    decode=DECODE,           # "greedy" o "beam"
)

Passant MolForge per 2000 fingerprints vàlides (model carregat una sola vegada)...
[2000/2000]


## 3. Guardar l'output

In [10]:
# Visualitzem el nou dataframe
df.head(5)

,SMILES_input,fingerprints_input_ECFP4,SMILES_output_ECFP4
id,,,
1,OCCCCN1CCCC1,80 222 422 473 541 646 653 740 743 774 807 810...,C1CCN(C1)CCCCN2CCN(P2(=NCCCCN3CCCC3)N4CCCC4)CC...
2,N#CN(CC(N)=O)c1ccc(Cl)cc1-c1cccnn1,46 73 80 140 165 212 216 218 281 322 352 353 3...,C1=CC(=NN=C1)C2=C(C=CC(=C2)Cl)N(CN(C#N)N(CC(=O...
3,Cc1cc[n+](C)cc1-c1ccccc1,106 222 263 350 352 389 391 440 463 482 522 67...,CC1=C(C=[N+](C=C1)C)C2=C(C=[N+](C=C2)C)C3=C(C=...
4,COC(=O)NC1CCN(C(=O)CCCCC(=O)N2CCCCC2)C1,2 16 80 129 194 202 222 369 387 433 505 526 65...,COC(=O)N[C@H]1CCN(C1)C(=O)CCCCC(=O)N2CCC(C2)NN...
5,COc1cccc(S(=O)(=O)NCC(=O)OC=CC2=CC=C2C(=O)Nc2c...,13 80 102 155 191 198 261 263 314 317 319 322 ...,CN1CCC(=C1/C=C/OC(=O)CNS(=O)(=O)C2=CC=CC(=C2)O...


In [11]:
df.to_csv(output_path)